# Avaliação final no conjunto de teste

Carrega `models/malware_random_forest_latest.joblib`, recria a divisão estratificada 70/15/15 com `random_state=42` (igual ao pipeline) e gera previsões para **todo o conjunto de teste**. `Malicious` fica separado e só é usado na avaliação.


> **Importante:** para manter o teste independente, o modelo avaliado tem de ter sido treinado apenas com treino. O modelo `latest` atual foi guardado pelo notebook 04, que ajusta o estimador no treino. Modelos guardados pelo notebook 03 incluem treino e validação.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import importlib
import inspect
import json
import hashlib
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
sys.modules.pop("data_preparation", None)
import data_preparation as dp
dp = importlib.reload(dp)
if Path(dp.__file__).resolve() != (ROOT / 'src' / 'data_preparation.py').resolve():
    raise ImportError(f'data_preparation carregado do caminho inesperado: {dp.__file__}')
if 'manifest_path' not in inspect.signature(dp.run_data_preparation).parameters:
    raise RuntimeError('Reinicia o kernel e recarrega este notebook para usar data_preparation.py atualizado.')
print(f'data_preparation ativo: {dp.__file__}')

DATA_PATH = ROOT / 'dataset' / 'raw' / 'Malware_and_benign_recognition.csv'
MODEL_PATH = ROOT / 'models' / 'malware_random_forest_latest.joblib'
METADATA_PATH = ROOT / 'models' / 'malware_random_forest_latest.json'
SPLIT_MANIFEST_PATH = ROOT / 'dataset' / 'processed' / 'split_manifest.csv'
OUTPUT_DIR = ROOT / 'models'
RANDOM_STATE = 42
TARGET = 'Malicious'
POSITIVE_LABEL = 1


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset não encontrado: {DATA_PATH}')
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Modelo latest não encontrado: {MODEL_PATH}')

df = pd.read_csv(DATA_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8')) if METADATA_PATH.exists() else {}
if metadata.get('preparation_version') != dp.PREPARATION_VERSION:
    raise ValueError('O modelo latest foi treinado com outra versão da preparação. Executa novamente o notebook 04 antes de avaliar o teste.')
split_provenance = metadata.get('split_manifest', {})
current_manifest_hash = hashlib.sha256(SPLIT_MANIFEST_PATH.read_bytes()).hexdigest()
if split_provenance.get('sha256') != current_manifest_hash:
    raise ValueError('O modelo latest nao foi treinado com o manifesto ativo. Executa novamente o notebook 04 antes de avaliar o teste.')
if split_provenance.get('training_partitions') != ['train']:
    raise ValueError('O modelo latest nao declara treino exclusivo no grupo train. Executa novamente o notebook 04 antes de avaliar o teste.')
feature_names = metadata.get('features')
if not feature_names:
    raise ValueError('Os metadados do modelo não incluem a lista de features.')
missing = sorted(set(feature_names) - set(df.columns))
if missing:
    raise ValueError(f'Faltam features necessárias no dataset: {missing}')
if TARGET not in df.columns:
    raise ValueError(f'Falta a coluna alvo {TARGET}.')

# Aplica a particao permanente; o target não entra em X_test.
X_train, X_val, X_test, y_train, y_val, y_test = dp.run_data_preparation(
    str(DATA_PATH), target=TARGET, random_state=RANDOM_STATE,
    manifest_path=SPLIT_MANIFEST_PATH,
)
print(f'Amostras de teste: {len(X_test)}')

In [ ]:
model = joblib.load(MODEL_PATH)
model_features = list(getattr(model, 'feature_names_in_', feature_names))
if model_features != list(feature_names):
    raise ValueError('As features dos metadados não correspondem à ordem esperada pelo modelo.')
X_test = X_test.loc[:, feature_names]

threshold_info = metadata.get('metricas', {}).get('threshold_selection', {})
threshold = threshold_info.get('threshold', 0.5)
classes = list(model.classes_)
positive_index = classes.index(POSITIVE_LABEL)
if hasattr(model, 'predict_proba'):
    scores = model.predict_proba(X_test)[:, positive_index]
    score_name = 'probabilidade_malicioso'
elif hasattr(model, 'decision_function'):
    scores = model.decision_function(X_test)
    if getattr(scores, 'ndim', 1) == 1 and classes[1] != POSITIVE_LABEL:
        scores = -scores
    elif getattr(scores, 'ndim', 1) > 1:
        scores = scores[:, positive_index]
    threshold = threshold_info.get('threshold', 0.0)
    score_name = 'score_malicioso'
else:
    raise TypeError('O modelo não fornece predict_proba nem decision_function.')

y_pred = (scores >= threshold).astype(int)
report = pd.DataFrame({
    'File': df.loc[X_test.index, 'File'].values if 'File' in df.columns else X_test.index,
    'Malicious_real': y_test.astype(int).values,
    'Malicious_previsto': y_pred,
    score_name: scores,
    'correto': y_test.astype(int).values == y_pred,
}, index=X_test.index)
report['classe_prevista'] = report['Malicious_previsto'].map({0: 'Benigno', 1: 'Malicioso'})
report

In [ ]:
print(f'Modelo: {metadata.get("modelo", MODEL_PATH.name)}')
print(f'Samples no teste: {len(report)} | Limiar: {threshold:.4f}')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 macro: {f1_score(y_test, y_pred, average="macro"):.4f}')
print('Matriz de confusão [real 0/1, previsto 0/1]:')
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
print('Relatório por classe:')
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=['Benigno', 'Malicioso'], zero_division=0))

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
report_path = OUTPUT_DIR / f'relatorio_teste_malware_{timestamp}.csv'
report.to_csv(report_path, index=False)
print(f'Relatório do conjunto de teste guardado em: {report_path}')